$\mathcal{G}:$
```
          U1     U2       U3
        ↙    ↘↙    ↘  ↙    ↘
       W ───→ Z ───→ X ────→ Y 
       ↑      ↑      ↑      
       │      U5     │
       │      ↓      │      
       U4────→M──────
```

W->Z, Z->X, X->Y, M->X

U1->W, U1->Z, U2->Z, U2->X, U3->X, U3->Y, U4->W, U4->M, U5->M, U5->Z

In [26]:
import numpy as np
import pandas as pd
%load_ext autoreload
%autoreload 2

from autobound.causalProblem import causalProblem
from autobound.DAG import DAG
from autobound.Query import Query

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


We have $\theta=(Y=1|do(X=1))$.

The mission of this notebook:
Calculate 
1. $pot(W,Z,X|do(M=1))$
2. $pot(M,Z,X|do(W=1))$


## Step 0: Simulate SCM

In [27]:
N_samples = 1000000
np.random.seed(42)

# We are simulating a ground truth SCM. That is, we will 
# 1. randomly pick all root variables, 
# 2. sample other variables as some arbitrary boolean function of their parents.
# We will simulate the actual confounders (not response types!) as binary variables.

# 1. Simulate Root variables (all unobserved confounders)
U1 = np.random.binomial(1, 0.4, N_samples)
U2 = np.random.binomial(1, 0.6, N_samples)
U3 = np.random.binomial(1, 0.3, N_samples)
U4 = np.random.binomial(1, 0.5, N_samples)
U5 = np.random.binomial(1, 0.7, N_samples)

# 2. Structural equations
W = U1 ^ U4
M = U4 & U5
Z = (W & U2) ^ (U1 | U5)
X = (Z ^ U2) | (M & U3)
Y = X ^ U3

ground_truth_data = pd.DataFrame({
    'W': W,
    'Z': Z,
    'X': X,
    'Y': Y,
    'M': M,
    'U1': U1,
    'U2': U2,
    'U3': U3,
    'U4': U4,
    'U5': U5
})

## Step 1: Compute $\mathcal{W}_\emptyset$

In [28]:
# Observational data
obs = ground_truth_data.drop(columns=['U1', 'U2', 'U3', 'U4', 'U5'])
obs_counts = obs.value_counts().reset_index(name='count')

# Round probabilities and force exact normalization after rounding
obs_counts['prob'] = (obs_counts['count'] / N_samples).round(6)
diff = round(1.0 - obs_counts['prob'].sum(), 6)

if diff != 0:
    idx = obs_counts['count'].idxmax()
    obs_counts.loc[idx, 'prob'] = round(obs_counts.loc[idx, 'prob'] + diff, 6)

obs_counts.drop(columns=['count']).to_csv('data/figure-15-obs.csv', index=False)

In [29]:
obs_counts.drop(columns=['count'])

,W,Z,X,Y,M,prob
0,0,1,0,0,0,0.113389
1,1,0,1,1,1,0.088216
2,1,0,1,1,0,0.084413
3,0,1,1,1,0,0.075112
4,0,1,0,0,1,0.058872
5,1,1,1,1,1,0.058578
6,1,1,1,1,0,0.056188
7,0,1,0,1,0,0.048643
8,0,1,1,0,1,0.042096
9,0,1,1,1,1,0.039085


In [30]:
dag = DAG()
dag.from_structure("W->Z, Z->X, X->Y, M->X, " \
"U1->W, U1->Z, U2->Z, U2->X, U3->X, U3->Y, U4->W, U4->M, U5->M, U5->Z", unob='U1,U2,U3,U4,U5')
problem = causalProblem(dag)
problem.load_data('data/figure-15-obs.csv')
problem.add_prob_constraints()
problem.set_estimand(problem.query('Y(X=1)=1'))
program = problem.write_program()

(lb, ub) = program.run_pyomo('couenne', verbose=False, parallel=True)
print(f"P(θ)∈[{lb:.4f}, {ub:.4f}]")
calW_empty = ub - lb
print(f"calW_empty = {calW_empty:.8f}")

P(θ)∈[0.4393, 0.7863]
calW_empty = 0.34700000


## Step 2: Compute $\mathcal{W_{a}}$

### Algorithm: Approximating $W_a(p_a)$ via Grid Search

We want to compute the worst-case width $W_a = \max_{p_a} W(p_a)$ of the identified set
for $\theta = P(Y=1 | do(X=1))$, maximized over all possible experimental outcomes $p_a$.

Since we don't know the true interventional distribution $p_a$, we search over all
candidate probability vectors that could plausibly arise from the experiment.

1. **Bound the search space** using observational data (Tian & Pearl, 2000, eq. 22):
   For each outcome tuple, the interventional probability is bounded by:
     - Lower bound: $P(\text{interv\_var}=v, \text{outcome})$
     - Upper bound: $1 - P(\text{interv\_var}=v) + P(\text{interv\_var}=v, \text{outcome})$

2. **Discretize** each bounded interval into a grid with step size $1/n$.

3. **Enumerate** all grid vectors that sum to 1 and fall within the bounds.

4. **For each candidate vector**, solve the partial identification problem and record the width.

5. **Return the maximum width** across all feasible candidate vectors.
   The potency is then $pot(a) = W_\emptyset - W_a$.

### Illustration: True interventions

In [31]:
def getIntervention_doM(M_val):
    X_doM = (Z ^ U2) | (M_val & U3)
    doM = pd.DataFrame({'W_doM': W, 'Z_doM': Z, 'X_doM': X_doM})
    doM_counts = doM.value_counts().reset_index(name='count')
    doM_counts['prob'] = (doM_counts['count'] / N_samples).round(6)
    return doM_counts.drop(columns=['count'])

def getIntervention_doW(W_val):
    Z_doW = (W_val & U2) ^ (U1 | U5)
    X_doW = (Z_doW ^ U2) | (M & U3)
    doW = pd.DataFrame({'M_doW': M, 'Z_doW': Z_doW, 'X_doW': X_doW})
    doW_counts = doW.value_counts().reset_index(name='count')
    doW_counts['prob'] = (doW_counts['count'] / N_samples).round(6)
    return doW_counts.drop(columns=['count'])

print("True P(W,Z,X | do(M=1)):")
print(getIntervention_doM(1))
print("\nTrue P(M,Z,X | do(W=1)):")
print(getIntervention_doW(1))

True P(W,Z,X | do(M=1)):
   W_doM  Z_doM  X_doM      prob
0      1      0      1  0.257206
1      0      1      1  0.237246
2      1      1      1  0.180037
3      0      1      0  0.172261
4      0      0      1  0.064822
5      1      1      0  0.038056
6      1      0      0  0.025213
7      0      0      0  0.025159

True P(M,Z,X | do(W=1)):
   M_doW  Z_doW  X_doW      prob
0      0      0      1  0.282465
1      1      0      1  0.210425
2      0      1      1  0.187628
3      1      1      1  0.139418
4      0      1      0  0.108167
5      0      0      0  0.071897


In [32]:
from itertools import product

def generate_feasible_vectors(n, bounds_list):
    """Generate all probability vectors on grid 1/n, summing to 1, within bounds."""
    k = len(bounds_list)

    def recurse(idx, remaining, current):
        if idx == k - 1:
            val = remaining / n
            lb, ub = bounds_list[idx]
            if lb - 1e-9 <= val <= ub + 1e-9:
                yield current + [val]
            return
        lb, ub = bounds_list[idx]
        lo = max(int(np.ceil(lb * n)), 0)
        hi = min(int(np.floor(ub * n)), remaining)
        for a in range(lo, hi + 1):
            yield from recurse(idx + 1, remaining - a, current + [a / n])

    return list(recurse(0, n, []))

### $pot(W,Z,X | do(M=1))$

In [33]:
def W_doM(p_config_0, p_config_1):
    problem = causalProblem(dag)
    problem.load_data('data/figure-15-obs.csv')
    problem.add_prob_constraints()
    problem.set_estimand(problem.query('Y(X=1)=1'))

    for M_val, p_config in [(0, p_config_0), (1, p_config_1)]:
        for _, row in p_config.iterrows():
            w, z, x = int(row.W_doM), int(row.Z_doM), int(row.X_doM)
            lhs = problem.query(f'W(M={M_val})={w}&Z(M={M_val})={z}&X(M={M_val})={x}')
            problem.add_constraint(lhs - Query(row.prob))
    program = problem.write_program()
    (lb, ub) = program.run_pyomo('couenne', verbose=False, parallel=True)
    return ub - lb

In [34]:
true_doM0 = getIntervention_doM(0)
true_doM1 = getIntervention_doM(1)
print("True P(W,Z,X | do(M=0)):")
print(true_doM0)
print("\nTrue P(W,Z,X | do(M=1)):")
print(true_doM1)

W_doM_true = W_doM(true_doM0, true_doM1)
print(f"\nW_doM with true p = {W_doM_true:.8f}")
print(f"pot(W,Z,X|do(M))  = {calW_empty - W_doM_true:.8f}")

True P(W,Z,X | do(M=0)):
   W_doM  Z_doM  X_doM      prob
0      1      0      1  0.246535
1      0      1      0  0.246355
2      1      1      1  0.163894
3      0      1      1  0.163152
4      1      1      0  0.054199
5      0      0      1  0.053968
6      0      0      0  0.036013
7      1      0      0  0.035884

True P(W,Z,X | do(M=1)):
   W_doM  Z_doM  X_doM      prob
0      1      0      1  0.257206
1      0      1      1  0.237246
2      1      1      1  0.180037
3      0      1      0  0.172261
4      0      0      1  0.064822
5      1      1      0  0.038056
6      1      0      0  0.025213
7      0      0      0  0.025159

W_doM with true p = 0.34700230
pot(W,Z,X|do(M))  = -0.00000230


### $pot(M,Z,X | do(W=1))$

In [35]:
def W_doW(p_config_0, p_config_1):
    problem = causalProblem(dag)
    problem.load_data('data/figure-15-obs.csv')
    problem.add_prob_constraints()
    problem.set_estimand(problem.query('Y(X=1)=1'))

    for W_val, p_config in [(0, p_config_0), (1, p_config_1)]:
        for _, row in p_config.iterrows():
            m, z, x = int(row.M_doW), int(row.Z_doW), int(row.X_doW)
            lhs = problem.query(f'M(W={W_val})={m}&Z(W={W_val})={z}&X(W={W_val})={x}')
            problem.add_constraint(lhs - Query(row.prob))
    program = problem.write_program()
    (lb, ub) = program.run_pyomo('couenne', verbose=False, parallel=True)
    return ub - lb

In [36]:
true_doW0 = getIntervention_doW(0)
true_doW1 = getIntervention_doW(1)
print("True P(M,Z,X | do(W=0)):")
print(true_doW0)
print("\nTrue P(M,Z,X | do(W=1)):")
print(true_doW1)

W_doW_true = W_doW(true_doW0, true_doW1)
print(f"\nW_doW with true p = {W_doW_true:.8f}")
print(f"pot(M,Z,X|do(W))  = {calW_empty - W_doW_true:.8f}")

True P(M,Z,X | do(W=0)):
   M_doW  Z_doW  X_doW      prob
0      0      1      0  0.282465
1      1      1      1  0.202755
2      0      1      1  0.187628
3      1      1      0  0.147088
4      0      0      1  0.108167
5      0      0      0  0.071897

True P(M,Z,X | do(W=1)):
   M_doW  Z_doW  X_doW      prob
0      0      0      1  0.282465
1      1      0      1  0.210425
2      0      1      1  0.187628
3      1      1      1  0.139418
4      0      1      0  0.108167
5      0      0      0  0.071897

W_doW with true p = 0.34700235
pot(M,Z,X|do(W))  = -0.00000235


In [37]:
print(f"W_empty = {calW_empty:.8f}")
print(f"W_doM   = {W_doM_true:.8f}, W_doW   = {W_doW_true:.8f}")
print(f"pot(W,Z,X|do(M))={calW_empty - W_doM_true:.8f}")
print(f"pot(M,Z,X|do(W))={calW_empty - W_doW_true:.8f}")

W_empty = 0.34700000
W_doM   = 0.34700230, W_doW   = 0.34700235
pot(W,Z,X|do(M))=-0.00000230
pot(M,Z,X|do(W))=-0.00000235
